# Embeddings com BERT em Português

Neste notebook, exploramos como diferentes modelos baseados em BERT podem representar sentenças em português.  
O objetivo é compreender como transformar texto em **vetores numéricos** (embeddings) e, a partir deles, aplicar técnicas de **similaridade**, **clusterização** e **visualização 2D**.  

Essa abordagem nos permite investigar como modelos multilíngues (mBERT) e monolíngues (BERTimbau, BERTugues) capturam relações semânticas entre frases simples do cotidiano.


### Instalação de Bibliotecas Necessárias

Antes de começarmos a trabalhar com embeddings e modelos de linguagem, precisamos instalar algumas bibliotecas que não vêm por padrão no ambiente.

O comando abaixo instala:

- **transformers**: biblioteca da Hugging Face para carregar modelos de linguagem como BERT, GPT etc.
- **sentence-transformers**: implementações otimizadas de modelos para embeddings semânticos de sentenças.
- **scikit-learn**: pacote essencial para aprendizado de máquina em Python, incluindo funções de clustering (ex.: KMeans), redução de dimensionalidade (ex.: PCA) e métricas.

In [ ]:
!pip install transformers datasets sentence-transformers scikit-learn

### Importação das Bibliotecas

Nesta etapa, organizamos os **imports** necessários para o projeto.  
Eles estão agrupados por finalidade para facilitar a leitura e o entendimento:

- **Numérico e dados**
  - `numpy` e `pandas`: manipulação de vetores, matrizes e dataframes.
  - `torch`: utilizado para trabalhar com tensores e rodar os modelos de linguagem da Hugging Face (com GPU se disponível).

- **Transformers / Hugging Face**
  - `AutoTokenizer` e `AutoModel`: classes que permitem carregar facilmente diferentes modelos pré-treinados de linguagem e seus tokenizadores.

- **Scikit-learn**
  - `KMeans`: algoritmo de clustering (agrupamento não supervisionado).
  - `PCA`: técnica de redução de dimensionalidade para visualização ou pré-processamento.
  - `cosine_similarity`: cálculo da similaridade semântica entre embeddings.

- **Visualização**
  - `matplotlib.pyplot`: biblioteca clássica para visualização estática.
  - `plotly.express` e `plotly.graph_objects`: visualizações interativas, como scatterplots e heatmaps.

Por fim, definimos a variável `seed = 42`, que serve como **semente aleatória** para garantir reprodutibilidade em algoritmos estocásticos (como PCA ou KMeans).  


In [ ]:
# Numérico e dados
import numpy as np
import pandas as pd
import torch

# Transformers / Hugging Face
from transformers import AutoTokenizer, AutoModel

# Scikit-learn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Visualização
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go


seed = 42


### Funções Auxiliares: Preparação e Codificação das Sentenças

Nesta parte, criamos funções essenciais para trabalhar com embeddings de modelos baseados em BERT.

1. **`get_device()`**  
   Verifica se há uma GPU disponível.  
   - Se sim, retorna `"cuda"`.  
   - Caso contrário, usa `"cpu"`.  
   Isso garante que o código rode de forma eficiente em diferentes ambientes (Colab, PC pessoal, servidor com GPU, etc.).

2. **`mean_pooling(last_hidden_state, attention_mask)`**  
   Implementa o *mean pooling* manualmente:
   - O BERT gera uma matriz `last_hidden_state` com as representações de cada token (dimensão `[B, T, H]`, onde *B* é o batch, *T* o número de tokens e *H* o tamanho do embedding).  
   - Para calcular a média apenas sobre os tokens válidos (sem contar *padding*), usamos a `attention_mask`.  
   - O resultado é um vetor por sentença, representando a média dos embeddings de seus tokens.

3. **`encode_sentences(model_name, sentences, pooling="cls", batch_size=32)`**  
   Função que transforma sentenças em embeddings numéricos:
   - Carrega o **tokenizer** e o **modelo** a partir do nome informado (`model_name`).  
   - Para cada lote de sentenças (`batch`):
     - Aplica o tokenizador.  
     - Passa os tokens pelo modelo, obtendo `last_hidden_state`.  
     - Extrai os embeddings da forma escolhida em `pooling`:  
       - `"cls"`: usa apenas o primeiro token especial `[CLS]`, que funciona como uma “resumo da sentença”.  
       - `"mean"`: usa a média dos embeddings de todos os tokens válidos (via `mean_pooling`).  
   - Retorna uma matriz `[N, H]`, em que cada linha é o embedding de uma sentença.

   Obs.: a função é decorada com `@torch.no_grad()`, evitando o cálculo de gradientes (não faremos *fine-tuning* aqui, apenas inferência).

4. **`reduce_2d(X, method="pca", random_state=seed)`**  
   Reduz a dimensionalidade dos embeddings para 2 componentes principais, permitindo **visualizações em 2D**:  
   - Por padrão, usa **PCA** (*Principal Component Analysis*).  
   - Retorna as coordenadas reduzidas `Z` e a variância explicada pelos dois primeiros componentes.  
   - Se outro método for solicitado, gera um erro (a função pode ser expandida no futuro para incluir UMAP, t-SNE etc.).

Essas funções formam a base do pipeline: carregar embeddings, escolher a estratégia de pooling, e reduzir a dimensionalidade para visualização ou análise.


---



### Pooling em Modelos BERT: CLS vs. Mean

Quando usamos o BERT (ou modelos derivados) para gerar embeddings de sentenças, precisamos decidir **como transformar a sequência de embeddings de tokens em um único vetor representativo da frase**.  
Esse processo é chamado de **pooling**.

#### 1. O token especial `[CLS]`
- O BERT adiciona, no início de cada sequência, um **token especial chamado `[CLS]`** (*classification token*).  
- Durante o treinamento original, esse token foi ajustado para capturar uma **representação global da sentença**, já que era usado em tarefas de classificação (ex.: “essa frase é positiva ou negativa?”).  
- Assim, um método simples é usar **apenas o embedding do `[CLS]`** como vetor da frase.  
  - Vantagem: rápido, já vem pronto no modelo.  
  - Limitação: pode não capturar nuances semânticas em sentenças mais longas ou complexas.

#### 2. Pooling por média (*mean pooling*)
- Outra abordagem é calcular a **média dos embeddings de todos os tokens da frase**.  
- Para isso, ignoramos os tokens de *padding* usando a `attention_mask`.  
- O resultado é um vetor que combina a informação distribuída em todos os tokens.  
  - Vantagem: tende a ser mais robusto e informativo em tarefas de similaridade semântica.  
  - Limitação: pode diluir informações importantes se a sentença for muito longa.

#### 3. Comparação resumida
- **CLS pooling**:  
  - Usa só o vetor `[CLS]`.  
  - Mais rápido.  
  - Bom para classificação.  

- **Mean pooling**:  
  - Usa a média de todos os tokens válidos.  
  - Mais informativo para medir similaridade semântica.  
  - Usado em modelos como *Sentence-BERT*.  

Em resumo:  
- Para **classificação de sentenças**, o pooling por `[CLS]` costuma ser suficiente.  
- Para **tarefas de similaridade semântica e clustering**, o pooling por **mean** geralmente gera resultados melhores.  


In [ ]:
def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def mean_pooling(last_hidden_state, attention_mask):
    # last_hidden_state: [B, T, H]; attention_mask: [B, T]
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

@torch.no_grad()
def encode_sentences(model_name, sentences, pooling="cls", batch_size=32):
    """
    pooling: 'cls' ou 'mean'
    """
    device = get_device()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_vecs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        outputs = model(**inputs)  # last_hidden_state: [B, T, H]
        if pooling == "cls":
            vecs = outputs.last_hidden_state[:, 0, :]                       # [B, H]
        elif pooling == "mean":
            vecs = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])  # [B, H]
        else:
            raise ValueError("pooling deve ser 'cls' ou 'mean'")
        all_vecs.append(vecs.detach().cpu().numpy())

    X = np.vstack(all_vecs)
    return X  # shape: [N, H]

def reduce_2d(X, method="pca", random_state=seed):
    if method.lower() == "pca":
        reducer = PCA(n_components=2, random_state=random_state)
        Z = reducer.fit_transform(X)
        expl = reducer.explained_variance_ratio_
        return Z, ("PCA", expl)
    else:
        raise ValueError("method deve ser 'pca'")


### Conjunto de Sentenças e Modelos Utilizados

1. **Sentenças de Exemplo**
   Definimos uma pequena lista de frases em português (`sentencas`) que servirão como insumo para gerar embeddings.  
   Essas frases são simples e variadas, incluindo:
   - Ações humanas cotidianas: *"Ele gosta de jogar futebol."*, *"Ela joga tênis nos finais de semana."*  
   - Situações envolvendo animais: *"O gato está dormindo no sofá."*, *"O cachorro latiu alto no quintal."*  
   - Descrições do ambiente: *"O céu está azul e claro hoje."*, *"A previsão do tempo indica céu ensolarado."*  

   Esse conjunto reduzido facilita a demonstração prática de como diferentes modelos capturam semântica e similaridade.

2. **Modelos Escolhidos**
   Criamos um dicionário (`modelos`) que associa nomes amigáveis a identificadores de modelos hospedados na Hugging Face:

   - **mBERT** (`bert-base-multilingual-cased`)  
     Modelo multilíngue treinado em 104 idiomas, incluindo português.  
     Bom para tarefas que envolvem múltiplas línguas, mas tende a ter desempenho inferior em português em comparação com modelos monolíngues.

   - **BERTimbau** (`neuralmind/bert-base-portuguese-cased`)  
     Modelo específico para o português brasileiro, treinado em grandes corpora no idioma.  
     Geralmente apresenta melhor desempenho em tarefas de PLN em português.

   - **BERTugues** (`ricardoz/BERTugues-base-portuguese-cased`)  
     Outra iniciativa de modelo monolíngue em português, com vocabulário e pesos ajustados para o idioma.  
     Permite comparar variações de treinamento dedicadas ao português.

👉 Dessa forma, poderemos comparar como modelos multilíngues e monolíngues se comportam ao gerar embeddings para as mesmas frases.


In [ ]:
# textos

sentencas = [ "Ele gosta de jogar futebol.",
              "Ela joga tênis nos finais de semana.",
              "O gato está dormindo no sofá.",
              "O cachorro latiu alto no quintal.",
              "O céu está azul e claro hoje.",
              "A previsão do tempo indica céu ensolarado."
            ]


modelos = {
    "mBERT": "bert-base-multilingual-cased",
    "BERTimbau": "neuralmind/bert-base-portuguese-cased",
    "BERTugues": "ricardoz/BERTugues-base-portuguese-cased"
}


### Geração dos Embeddings

Agora, vamos utilizar os modelos definidos anteriormente para transformar cada sentença em um **vetor numérico** (embedding).  
Os embeddings permitem medir similaridade semântica entre frases, bem como realizar agrupamentos (clustering).

1. **Estrutura de Armazenamento**
   - Criamos um dicionário chamado `embeds`.  
   - A chave é uma tupla `(modelo, pooling)` — por exemplo, `("BERTimbau", "CLS")`.  
   - O valor é uma matriz NumPy `[N, H]`, onde:
     - `N` = número de sentenças.  
     - `H` = dimensão do embedding (típico do modelo, ex.: 768).  

   ```python
   embeds = {}  # dict[(modelo, pooling)] -> np.ndarray [N, H]


In [ ]:
# gerar embeddings

embeds = {}  # dict[(modelo, pooling)] -> np.ndarray [N, H]

for nome_visivel, nome_hf in modelos.items():
    X_cls = encode_sentences(nome_hf, sentencas, pooling="cls")
    X_mean = encode_sentences(nome_hf, sentencas, pooling="mean")
    embeds[(nome_visivel, "CLS")] = X_cls
    embeds[(nome_visivel, "MEAN")] = X_mean

print({k: v.shape for k, v in embeds.items()})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{('mBERT', 'CLS'): (6, 768), ('mBERT', 'MEAN'): (6, 768), ('BERTimbau', 'CLS'): (6, 768), ('BERTimbau', 'MEAN'): (6, 768), ('BERTugues', 'CLS'): (6, 768), ('BERTugues', 'MEAN'): (6, 768)}


### Similaridade por Cosseno (heatmap) e Agrupamento (K-Means)

#### Heatmap
Calculamos a **similaridade cosseno** entre os embeddings (`cosine_similarity(X)`) para inspecionar proximidade semântica entre as sentenças.  
O heatmap exibe a matriz `S` (valores em `[0,1]`), com:
- **Máscara opcional** do triângulo superior (para evitar redundância).
- **Rótulos de célula** com os valores numéricos formatados.
- **Hover** que mostra os pares de sentenças (melhora a interpretabilidade).

> Interpretação: valores mais altos indicam sentenças com embeddings mais próximos angularmente (maior similaridade semântica).

---

#### Clusterização com K-Means
Aplicamos **K-Means** diretamente sobre os embeddings de cada modelo/pooling para **agrupar sentenças** por proximidade em espaço vetorial.

**Como funciona (essência):**
- Inicializa `k` centróides.
- Alterna entre:
  1) **Atribuição** de cada ponto ao centróide mais próximo (distância euclidiana).  
  2) **Atualização** dos centróides como média dos pontos em cada cluster.
- Converge quando as atribuições/centróides estabilizam (ou atinge o limite de iterações).

**Decisões de implementação:**
- `n_clusters=3`: valor didático. Pode (e deve) ser calibrado empiricamente.
- `random_state=seed`: **reprodutibilidade** (mesmo particionamento ao repetir o experimento).
- `n_init='auto'`: versões recentes do scikit-learn usam múltiplas inicializações de forma adaptativa para reduzir o risco de mínimos locais ruins.

**Boas práticas e cuidados:**
- **Número de clusters** (`k`): avalie com *Elbow* (inércia), **silhouette score** ou **Davies–Bouldin**. Com `N` pequeno (como aqui), métricas podem oscilar; use mais dados para decisões robustas.
- **Escala dos dados**: K-Means usa **distância euclidiana**. Se modelos gerarem embeddings com escalas muito distintas, considere normalizar por modelo/pooling (ex.: `normalize(X)` ou `StandardScaler`) **antes** de agrupar.  
  > Observação: como comparamos *dentro* de cada (modelo, pooling), o impacto de escala entre modelos é minimizado.
- **Não reduza para 2D antes de clusterizar**: o K-Means deve rodar nos **embeddings originais** (alta dimensão). Projeções 2D (PCA/UMAP) são apenas para visualização; usar projeção para clusterização pode distorcer fronteiras.
- **Tamanho da amostra**: com apenas 6 sentenças, clusters são **instáveis** e mais sensíveis ao ruído.

**Interpretação do resultado:**
- A saída `labels` atribui cada sentença a um cluster `{0,1,2}`.  
- Juntamos `modelo`, `pooling`, `sentenca`, `cluster` em `clusters_df` para integrar com visualizações posteriores (scatter 2D facetado por modelo/pooling).

> Resumo: o heatmap oferece uma **leitura par-a-par** da similaridade; o K-Means fornece uma **partição global** das sentenças em grupos coerentes no espaço de embeddings.


In [ ]:

def plot_similarity_heatmap_px(
    X,
    sentences=None,          # lista de sentenças para o hover
    title="Similaridade (cosseno)",
    cmap="Blues",
    mask_upper=False,        # True: mostra só triângulo inferior
    vmin=0.0, vmax=1.0,
    cbar_label="Similaridade",
    width=700, height=600,
    fmt=".2f",               # formatação do label interno
    text_font_size=11,
    text_font_color="black",
    xgap=1, ygap=1           # “espessura” das linhas entre células
):
    # 1) Similaridade
    S = cosine_similarity(X)
    n = S.shape[0]
    labels = [f"S{i+1}" for i in range(n)]

    # 2) Máscara (triângulo superior)
    Z = S.astype(float).copy()
    if mask_upper:
        iu = np.triu_indices(n, k=1)
        Z[iu] = np.nan

    # 3) Labels numéricos dentro das células (vão como text)
    text_matrix = np.empty((n, n), dtype=object)
    text_matrix[:] = ""
    for i in range(n):
        for j in range(n):
            if not np.isnan(Z[i, j]):
                text_matrix[i, j] = f"{Z[i, j]:{fmt}}"

    # 4) Hover somente com as sentenças (sem número)
    #    Usamos customdata com HTML simples para melhor legibilidade
    customdata = np.empty((n, n), dtype=object)
    customdata[:] = ""
    for i in range(n):
        for j in range(n):
            if not np.isnan(Z[i, j]):
                s1 = sentences[i] if sentences is not None else labels[i]
                s2 = sentences[j] if sentences is not None else labels[j]
                customdata[i, j] = f"<b>{labels[i]}</b>: {s1}<br><b>{labels[j]}</b>: {s2}"

    # 5) Construir heatmap
    fig = go.Figure(
        data=go.Heatmap(
            z=Z,
            x=labels,
            y=labels,
            zmin=vmin, zmax=vmax,
            colorscale=cmap,
            colorbar=dict(title=cbar_label),
            # labels internos
            text=text_matrix,
            texttemplate="%{text}",
            textfont=dict(color=text_font_color, size=text_font_size),
            # hover apenas com as frases
            customdata=customdata,
            hovertemplate="%{customdata}<extra></extra>",
            # “grades” entre células
            xgap=xgap, ygap=ygap
        )
    )

    # Layout
    fig.update_layout(
        title=title,
        width=width, height=height,
        template="plotly_white",
        margin=dict(l=60, r=30, t=60, b=60),
    )
    # Células quadradas e origem no topo
    fig.update_yaxes(autorange="reversed", scaleanchor="x", scaleratio=1)

    fig.show()
    return S, fig


def run_kmeans(X, n_clusters=3, random_state=seed):
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    labels = km.fit_predict(X)
    return labels


# ===== Uso =====
resultados = []

for (modelo, pooling), X in embeds.items():
    # Similaridade (agora desempacotando corretamente)
    S, _ = plot_similarity_heatmap_px(
        X,
        sentences=sentencas,  # <- inclui textos no hover
        title=f"Similaridade ({modelo}, {pooling})",
        cmap="Blues",
        mask_upper=False,     # ou True, se quiser só triângulo inferior
        vmin=0.0, vmax=1.0,
        cbar_label="Similaridade",
        fmt=".2f",
        xgap=1, ygap=1
    )
    # Clustering
    labels = run_kmeans(X, n_clusters=3, random_state=seed)
    resultados.append(pd.DataFrame({
        "modelo": modelo,
        "pooling": pooling,
        "sentenca": sentencas,
        "cluster": labels
    }))

clusters_df = pd.concat(resultados, ignore_index=True)

### Resultado da Clusterização

A tabela mostra o resultado da aplicação do algoritmo **K-Means** sobre os embeddings gerados pelo **mBERT**, utilizando as duas estratégias de pooling: `CLS` e `MEAN`.

Cada linha corresponde a uma sentença, com informações sobre:
- **modelo**: `mBERT` etc.
- **pooling**: estratégia de agregação usada (`CLS` ou `MEAN`).  
- **sentenca**: a frase original em português.  
- **cluster**: o rótulo atribuído pelo K-Means (0, 1 ou 2).  

#### Como interpretar
- Sentenças que receberam o **mesmo número de cluster** foram consideradas mais próximas no espaço vetorial.  
- Exemplo: *"Ele gosta de jogar futebol."* e *"Ela joga tênis nos finais de semana."* caíram no **cluster 2**, o que reflete a similaridade semântica entre ambas (atividades esportivas).  

Mesmo com poucas frases, já conseguimos observar como diferentes embeddings e estratégias de pooling podem organizar semanticamente as sentenças em grupos.

In [ ]:
clusters_df

,modelo,pooling,sentenca,cluster
0,mBERT,CLS,Ele gosta de jogar futebol.,2
1,mBERT,CLS,Ela joga tênis nos finais de semana.,2
2,mBERT,CLS,O gato está dormindo no sofá.,0
3,mBERT,CLS,O cachorro latiu alto no quintal.,0
4,mBERT,CLS,O céu está azul e claro hoje.,1
5,mBERT,CLS,A previsão do tempo indica céu ensolarado.,0
6,mBERT,MEAN,Ele gosta de jogar futebol.,2
7,mBERT,MEAN,Ela joga tênis nos finais de semana.,2
8,mBERT,MEAN,O gato está dormindo no sofá.,0
9,mBERT,MEAN,O cachorro latiu alto no quintal.,0


### Projeção 2D (PCA) e Dispersão Facetada por Modelo/Pooling

**Objetivo.** Reduzir embeddings de alta dimensão para **2 componentes principais** e visualizar, em um único painel, como **modelos** (colunas) e **estratégias de pooling** (linhas) organizam semanticamente as sentenças. As cores representam os **clusters** atribuídos pelo K-Means (rodado nos embeddings originais, não nos reduzidos).

#### 1) `reduce_2d(X, method="pca")`
- Utiliza **PCA** (*Principal Component Analysis*), técnica de projeção linear que preserva ao máximo a **variância global** dos dados nos dois primeiros eixos.
- Além das coordenadas reduzidas (`x`, `y`), retorna também a **variância explicada** — uma medida de quanta informação dos embeddings originais foi mantida.
- O código já prevê a possibilidade de usar **UMAP** futuramente, mas aqui focamos apenas no **PCA**.

#### 2) `plot_scatter_embeddings(...)`
- Para cada `(modelo, pooling)`, aplica a redução para 2D e cria um DataFrame com:
  - Coordenadas `x`, `y` (posição no plano).
  - Nome do modelo.
  - Estratégia de pooling (`CLS` ou `MEAN`).
  - Sentença original.  
- Em seguida, junta (`merge`) com o `clusters_df` para anexar o **rótulo de cluster** de cada sentença.
- Também gera um identificador curto `sid` (S1, S2, …) para facilitar a leitura no hover.

#### 3) Visualização interativa
- **Facetas**: colunas para os modelos e linhas para pooling → facilita a comparação direta.  
- **Cores**: cada cluster recebe uma cor discreta (tons de azul), permitindo verificar se os grupos são coerentes.  
- **Hover detalhado**: mostra `sid`, sentença, modelo, pooling, cluster e as coordenadas no plano 2D.  

#### Como interpretar
- Pontos **próximos no gráfico** indicam sentenças com embeddings similares.  
- Se esses pontos também compartilham a **mesma cor de cluster**, há consistência entre o agrupamento (K-Means) e a projeção (PCA).  
- Comparar colunas revela diferenças entre **mBERT, BERTimbau e BERTugues**.  
- Comparar linhas mostra o efeito de **CLS vs. MEAN pooling** na estruturação semântica.

Resumo: o scatter 2D é uma ferramenta de **exploração visual**, que ajuda a intuir como diferentes modelos e estratégias de pooling organizam sentenças no espaço vetorial.


In [ ]:
# Projeção 2D + scatter


def reduce_2d(X, method="pca", random_state=seed, n_neighbors=15, min_dist=0.1):
    method = method.lower()
    if method == "pca":
        reducer = PCA(n_components=2, random_state=random_state)
        Z = reducer.fit_transform(X)
        meta = ("PCA", reducer.explained_variance_ratio_)
        return Z, meta
    elif method == "umap":
        reducer = umap.UMAP(
            n_components=2,
            random_state=random_state,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            metric="euclidean",
        )
        Z = reducer.fit_transform(X)
        meta = ("UMAP", None)
        return Z, meta
    else:
        raise ValueError("method deve ser 'pca' ou 'umap'")

def plot_scatter_embeddings(embeds, sentences, method="pca", random_state=seed):
    """
    Gera um DataFrame 2D concatenando projeções por (modelo, pooling).
    embeds: dict[(modelo, pooling)] -> np.ndarray [N, H]
    sentences: lista de sentenças (mesmo N, ordem consistente)
    """
    rows = []
    for (modelo, pooling), X in embeds.items():
        Z, meta = reduce_2d(X, method=method, random_state=random_state)
        # Z: [N,2]
        df_tmp = pd.DataFrame({
            "x": Z[:, 0],
            "y": Z[:, 1],
            "modelo": modelo,
            "pooling": pooling,
            "sentenca": sentences,   # mantém coerência com clusters_df
        })
        rows.append(df_tmp)

    df2d = pd.concat(rows, ignore_index=True)
    return df2d

# --- 1) Preparar dados 2D (PCA ou UMAP) ---
df2d = plot_scatter_embeddings(embeds, sentences=sentencas, method="pca")

# Corrige a chave de merge: é 'sentenca' (singular), não 'sentencas'
df2d = df2d.merge(clusters_df, on=["modelo", "pooling", "sentenca"], how="left")

# ID curto para cada sentença (S1, S2, …)
df2d["sid"] = df2d["sentenca"].apply(lambda s: f"S{sentencas.index(s)+1}")

# --- 2) Converter 'cluster' para categórico (garante legenda discreta) ---
df2d["cluster"] = df2d["cluster"].astype(str)

# --- 3) Ordem explícita dos facetes ---
model_order = ["mBERT", "BERTimbau", "BERTugues"]
pool_order  = sorted(df2d["pooling"].unique())
cluster_order = sorted(df2d["cluster"].unique())

# --- 4) Paleta discreta para clusters (tons de azul) ---
k = int(df2d["cluster"].nunique())
blue_seq = px.colors.sequential.Blues
while len(blue_seq) < k:
    blue_seq = blue_seq + blue_seq
color_seq = blue_seq[-k:]

# --- 5) Scatter interativo com facetas e hover detalhado ---
fig = px.scatter(
    df2d,
    x="x", y="y",
    color="cluster",
    color_discrete_sequence=color_seq,
    facet_col="modelo",
    facet_row="pooling",
    facet_col_spacing=0.08,
    facet_row_spacing=0.10,
    category_orders={
        "modelo": model_order,
        "pooling": pool_order,
        "cluster": cluster_order
    },
    hover_data={
        "sid": True,
        "sentenca": True,
        "modelo": True,
        "pooling": True,
        "cluster": True,
        "x": ':.3f',
        "y": ':.3f'
    },
    title="Embeddings 2D por Modelo (colunas) e Pooling (linhas) — PCA"
)

fig.update_layout(
    template="plotly_white",
    legend_title_text="Cluster",
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(
    marker=dict(size=10, line=dict(width=0)),
    opacity=0.9,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"     # sid
        "Sentença: %{customdata[1]}<br>"
        "Modelo: %{customdata[2]} | Pooling: %{customdata[3]}<br>"
        "Cluster: %{customdata[4]}<br>"
        "x: %{x:.3f} | y: %{y:.3f}<extra></extra>"
    )
)

# Molduras ao redor de cada faceta (opcional)
for xaxis_name in [k for k in fig.layout if k.startswith("xaxis")]:
    xdom = getattr(fig.layout, xaxis_name).domain
    suffix = xaxis_name[5:]  # "" ou "2","3",...
    yaxis_name = "yaxis" + suffix
    if hasattr(fig.layout, yaxis_name):
        ydom = getattr(fig.layout, yaxis_name).domain
        fig.add_shape(
            type="rect",
            xref="paper", yref="paper",
            x0=xdom[0], x1=xdom[1], y0=ydom[0], y1=ydom[1],
            line=dict(color="rgba(0,0,0,0.28)", width=1),
            fillcolor="rgba(0,0,0,0)",
            layer="below"
        )

fig.show()


# Conclusão

Vimos como gerar embeddings de sentenças, comparar suas similaridades, agrupar em clusters e projetar os resultados em duas dimensões.  
Mesmo com um conjunto pequeno de frases, foi possível observar diferenças entre modelos e estratégias de pooling (`CLS` vs. `MEAN`).  

Este notebook encerra a primeira exploração de **contextual embeddings** em português, preparando terreno para estudos posteriores em tarefas como **parsing**, **extração de entidades** e **classificação de textos**.


---



### Pooling: teoria e evidências práticas

O artigo *“Enhancing Sentence Embedding with Generalized Pooling”* (Chen, Ling & Zhu, 2018) mostra que, embora o token `[CLS]` seja comum para construir a representação de toda a sentença, sua eficácia depende fortemente de como o modelo foi treinado (se foi ajustado para tarefas de sentença, etc.).  

Quando o modelo não tem supervisão direta para que `[CLS]` reflita bem todo o conteúdo da frase ou quando se quer usar embeddings “genéricos” para muitas tarefas, `mean pooling` ou variantes de pooling com atenção tendem a gerar representações mais estáveis e informativas. Ou seja:

- `CLS` é prático, barato computacionalmente (já vem com o modelo), bom para tarefas específicas (classificação se ajustado).
- `Mean pooling` reduz viés de seleção de token especial, tende a capturar melhor o contexto geral da frase, especialmente em tarefas de similaridade ou quando o modelo não foi supervisionado para usar CLS.  
